# Part 3 · Hamiltonians & Analog Quantum Computing

By the end of this notebook you will schedule four conference talks into two time slots with a quantum annealer, and read the answer directly off the final quantum state.

That is a real workload, reduced to teaching size. Quantum annealers (D-Wave's machines, and the analog QPUs that Qilimanjaro builds) are piloted today on exactly this family of problems: scheduling, logistics, portfolio selection. Here we build a miniature version of the pipeline by hand, so that every step is visible; Part 4 then automates the same pipeline for larger, messier problems.

In Part 2 we computed with **discrete gates**, the *digital* model, in which a program is a list of instructions. This part introduces the **analog** model, where the computation is one **continuous physical process** shaped by an energy function called a **Hamiltonian**. This is the native language of quantum annealers, and QiliSDK treats it as a first-class paradigm alongside circuits.

By the end of this notebook you will be able to:

- read a **Hamiltonian** as a *cost function over bitstrings*, and certify with plain numpy that its **ground state** (lowest-energy state) is the argmin;
- describe time-dependent control with a **`Schedule`** and evolve a state under it with the **`AnalogEvolution`** functional;
- run a complete **quantum-annealing** workflow on a real scheduling problem and decode the answer from the final state.

On the architecture map (`overall.jpg`), we move from the **Digital** primitives to the **Analog** ones (`Hamiltonian`, `Schedule`) and swap the functional to `AnalogEvolution`. The backend (`QiliSim`) and the `Readout` builder are the same ones you already know: the same `Backend.execute(functional, readout)` pattern, a different paradigm.

In [ ]:
# ▶ Run me first. No-op if QiliSDK is already installed; installs it on a fresh env (e.g. Google Colab).
try:
    import qilisdk
except ImportError:
    import sys
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "qilisdk[openqasm,qir]==0.2.1", "matplotlib", "numpy"], check=True)
    import qilisdk  # Colab: if this still fails, Runtime > Restart session, then rerun
print("QiliSDK", qilisdk.__version__)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## 3.0 · What is analog quantum computing?

Two one-line refreshers from Part 1, because everything below leans on them:

- **Hermitian matrix**: a matrix $O$ that equals its own conjugate transpose (`O == O.conj().T` in numpy). Hermitian matrices have real eigenvalues, which is what lets them represent measurable quantities.
- **Eigenvalue / eigenstate**: an eigenstate of $O$ is a state that $O$ only rescales, $O|\psi\rangle = \lambda|\psi\rangle$, and the eigenvalue $\lambda$ is that scale factor. Physically: a state whose measurement outcome is certain, and $\lambda$ is the value you will read.

Every quantum system has a **Hamiltonian** $H$: the Hermitian operator that represents its **energy**. The Hamiltonian does two jobs, and analog quantum computing exploits both.

**Job 1: it is a cost function.** The eigenvalues of $H$ are the energies the system is allowed to have. The smallest eigenvalue is the **ground-state energy**, and its eigenstate is the **ground state**: the argmin of the energy landscape. The central trick of this notebook is to *encode a problem so that the ground state is the answer*.

**Job 2: it generates motion.** The Schrödinger equation

$$i\hbar\,\frac{\partial}{\partial t}\,|\psi(t)\rangle = H(t)\,|\psi(t)\rangle$$

reads, in words: *the rate of change of the state equals the Hamiltonian applied to the state* (the constants $i\hbar$ handle units and phase). Here $|\psi(t)\rangle$ is the state at time $t$ and $\hbar$ ("h-bar") is just a units constant, which we set to $1$. For a constant $H$ the solution is $|\psi(t)\rangle = e^{-iHt}\,|\psi(0)\rangle$, where $e^{-iHt}$ is a **matrix exponential** (what `scipy.linalg.expm` computes): think of it as the continuous analog of applying one big gate, with $t$ as the knob for how much of it you apply.

A developer's framing: **digital** quantum computing executes a discrete instruction list, gate by gate, like a CPU stepping through opcodes. **Analog** quantum computing steers one continuous physical process, the way turning an analog knob differs from processing audio as DSP samples. It is the same physics underneath (a gate is a short, engineered burst of Hamiltonian evolution); what changes is the programming model.

## 3.1 · Building Hamiltonians from Pauli operators

The single-qubit **Pauli operators** are the alphabet Hamiltonians are written in. You met them as gates in Part 2; here they return in their other role, as *pieces of an energy function*:

$$I=\begin{pmatrix}1&0\\0&1\end{pmatrix},\quad X=\begin{pmatrix}0&1\\1&0\end{pmatrix},\quad Y=\begin{pmatrix}0&-i\\i&0\end{pmatrix},\quad Z=\begin{pmatrix}1&0\\0&-1\end{pmatrix}.$$

Products acting on different qubits, like $Z_0 Z_1$ ("$Z$ on qubit 0 times $Z$ on qubit 1"), are called **Pauli strings**, and every Hamiltonian can be written as a real-weighted sum of Pauli strings, the way every integer can be written in binary.

The workhorse example is the **Ising model**, physicists' standard model of interacting **spins**. A spin is a tiny magnet that points either up or down; we map up/down onto the bit values 0/1, which is exactly the $Z$ eigenvalue convention from Part 1: $Z|0\rangle = +1\,|0\rangle$ and $Z|1\rangle = -1\,|1\rangle$. The Ising Hamiltonian is

$$H = -\sum_{\langle i,j\rangle} J_{ij}\, Z_i Z_j \;-\; \sum_i h_i\, Z_i$$

where every symbol means something concrete:

- $\langle i,j\rangle$ ranges over the **connected pairs** of qubits (the edges of a graph: which spins can feel each other);
- $J_{ij}$ is the **coupling strength** for the pair $(i,j)$: with the leading minus sign, a positive $J_{ij}$ *rewards agreement* (lower energy when the two bits match) and a negative one *penalizes* it;
- $h_i$ is a **per-qubit bias**, a thumb on the scale that makes qubit $i$ individually prefer 0 or 1.

In QiliSDK, `from qilisdk.analog import X, Y, Z, I` gives you **factory functions**: `Z(0)` builds a one-term `Hamiltonian` acting on qubit 0, and ordinary `+`, `-`, `*` and scalar multiplication combine such terms. Products simplify automatically by the Pauli algebra (e.g. $X_1 Z_1 = -iY_1$).

> ⚠️ **Name collision.** These analog `X, Y, Z, I` are *Hamiltonian builders*, not the digital gate **classes** of the same names in `qilisdk.digital`. In a file that mixes both, alias on import (`from qilisdk.analog import Z as aZ`). This notebook is analog-only, so the bare names are safe.

One reconciliation with Part 1 before we build anything: an analog `Z(0)` and the numpy Pauli-$Z$ matrix you diagonalized in Part 1 are the *same object* in two wrappers, and `Readout().with_expectation(...)` (coming up in §3.2) is the executed-on-a-backend version of Part 1's `expect_val(operator, state)`. The next cell proves the first claim in one line.

In [ ]:
from qilisdk.analog import X, Y, Z, I, Hamiltonian

# Z(0) wraps the same matrix you used as numpy Pauli-Z in Part 1:
pauli_z = np.array([[1, 0], [0, -1]])
print("Z(0) equals Part 1's numpy Pauli-Z:", np.allclose(Z(0).to_qtensor().dense(), pauli_z))

# Combine factories with +, -, * and scalars -> a Hamiltonian
H = Z(0) + 0.5 * Z(0) * Z(1) + X(1)
print("H =", H)
print("acts on", H.nqubits, "qubits")

# The Pauli algebra simplifies products automatically:
print("X(1) * Z(1) =", X(1) * Z(1))          # -1j Y(1)

# .to_qtensor() gives the dense operator; 2 qubits -> a 4x4 matrix
print("matrix shape:", H.to_qtensor().shape)

### The key idea: a Hamiltonian is a cost function over bitstrings

Look at a Hamiltonian built only from $Z$s. In the computational basis $Z$ is diagonal, so a $Z$-only Hamiltonian is a **diagonal matrix**: it never mixes bitstrings, it just assigns each one a number. That number is the bitstring's **energy**, and the Hamiltonian is literally a lookup table `bitstring -> cost`.

You can even skip the matrix entirely. For any bitstring, each $Z_i$ contributes $+1$ if bit $i$ is `0` and $-1$ if it is `1` (the eigenvalue convention above), and each $Z_i Z_j$ contributes the product of those two values. That is a two-line Python function.

Take a tiny 2-qubit Ising Hamiltonian: one ferromagnetic coupling ($J_{01} = 1$, rewards agreement) plus a small bias toward 0 on both qubits ($h_0 = h_1 = 0.5$):

$$H_{\text{ising}} = -Z_0 Z_1 \;-\; 0.5\,(Z_0 + Z_1).$$

The next cell computes the energy of all four bitstrings *classically*, then cross-checks against numpy's eigensolver (`eigvalsh` is the variant for Hermitian matrices and returns eigenvalues in ascending order, so entry `[0]` is the ground-state energy). A hand-rolled loop agreeing with a linear-algebra library is the kind of cross-check worth trusting: we don't just claim the ground state is $|00\rangle$, we verify it.

In [ ]:
H_ising = -(Z(0) * Z(1)) - 0.5 * (Z(0) + Z(1))

def ising_energy(bits):
    """Classical energy of a bitstring: bit '0' -> z = +1, bit '1' -> z = -1."""
    z = [1 if b == "0" else -1 for b in bits]
    return -(z[0] * z[1]) - 0.5 * (z[0] + z[1])

print("bitstring | energy")
for bits in ["00", "01", "10", "11"]:
    print(f"   {bits}     | {ising_energy(bits):+.1f}")

# Cross-check: diagonalize the actual matrix
energies = np.linalg.eigvalsh(H_ising.to_qtensor().dense())
print()
print("eigvalsh energy levels:", np.round(energies, 3))          # [-2.  0.  1.  1.]
print("ground-state energy   :", round(float(energies[0]), 3))   # -2.0, the |00> row of the table

## 3.1b · The problem of the day: scheduling conference talks

EuroPython itself has this problem. Four talks must go into two time slots (morning and afternoon). Some pairs of talks **conflict**: they share a speaker or expect the same audience, so they must not run in the same slot. Assign every talk a slot such that no conflicting pair collides.

The whole encoding fits in three bullet points:

- **One qubit per talk.** Reading qubit $i$ as bit `0` means "talk $i$ runs in the morning", `1` means "afternoon". A 4-bit bitstring *is* a complete schedule.
- **One term per conflict.** For each conflicting pair $(i, j)$, add $Z_i Z_j$ to the Hamiltonian. By the table logic above: if talks $i$ and $j$ sit in the *same* slot, $z_i z_j = +1$ (an energy penalty); in *different* slots, $z_i z_j = -1$ (an energy reward).
- **A valid schedule is a minimum-energy bitstring.** Every conflict you resolve subtracts 1 from the energy, so the ground state resolves as many as physics allows (here: all of them).

This is exactly the Ising model with $J_{ij} = -1$ on the conflict edges and no bias field. Nothing quantum has happened yet, and that is the point: the encoding is pure bookkeeping, checkable with classical code. Here is the data:

In [ ]:
talks = ["Async Python", "Quantum 101", "Rust FFI", "Web Security"]
conflicts = [(0, 1), (1, 2), (2, 3), (0, 3)]   # pairs of talk indices that must not share a slot

for i, j in conflicts:
    print(f"conflict: {talks[i]!r} <-> {talks[j]!r}")

### 🧩 Exercise 3.1 · Encode the schedule, then certify it classically

**Goal:** build the problem Hamiltonian from `conflicts`, then prove by brute force that its minimum-energy bitstrings are exactly the two valid schedules.

**Why it matters:** this is the step every annealing project depends on. If the encoding is wrong, the annealer will faithfully find the ground state of the *wrong* cost function, and nothing downstream will flag the mistake. At 4 qubits you can still certify the encoding against all $2^4 = 16$ bitstrings; that becomes impossible around 50 qubits, so it is worth building the habit at toy size.

**Your task:**

1. Build `problem`: the sum of one $Z_i Z_j$ term per pair in `conflicts` (a generator expression inside `sum(...)` reads nicely).
2. Compute the energy of all 16 bitstrings. Two routes, pick either:
   - reuse the classical trick from the demo above (bit `0` contributes $z = +1$, bit `1` contributes $z = -1$, sum the products over the conflict pairs), or
   - read the whole table at once from the diagonal of the dense matrix with `np.diag(...)` (a $Z$-only Hamiltonian is diagonal).
3. Print each bitstring next to its energy. Remember Part 1's big-endian convention: basis index `k` corresponds to the bitstring `format(k, "04b")`, and the **first character is qubit 0**.
4. Report the minimum-energy bitstring(s). You should find exactly two, mirror images of each other. Sanity-check one of them against the conflict list by hand: do the four conflicting pairs really end up in opposite slots?

In [ ]:
n = len(talks)

# TODO 1: build `problem`, the sum of Z(i) * Z(j) over every pair (i, j) in conflicts

# TODO 2: compute all 16 energies (classical loop over bitstrings, or the diagonal
#         of the dense matrix: a Z-only Hamiltonian is diagonal)

# TODO 3: print a bitstring -> energy table
#         (basis index k <-> format(k, "04b"), first character = qubit 0)

# TODO 4: print the minimum-energy bitstring(s)

## 3.2 · Time-dependent control: the `Schedule`

We now have a cost function whose ground state is our answer. Job 2 of the Hamiltonian (generating motion) is how we will *reach* that ground state, and for that we need Hamiltonians that **change over time**.

A **`Schedule`** represents a time-dependent Hamiltonian as a sum of fixed Hamiltonians with time-varying **coefficient envelopes**:

$$H(t) = \sum_k c_k(t)\, H_k$$

where each $H_k$ is a fixed Hamiltonian (built like everything in §3.1) and each $c_k(t)$ is a plain Python function of time that scales it. You hand `Schedule` a dict of labeled Hamiltonians and, for each label, its coefficient function over a time interval.

The cell below builds the prototypical shape, over a total time $T$: a **driver** term $-X_0$ whose envelope ramps $1 \to 0$, and a **problem** term $Z_0$ whose envelope ramps $0 \to 1$. Writing $s = t/T$ for the **fraction of the run completed**, the driver's coefficient is $1 - s$ and the problem's is $s$.

Why $-X_0$ as the starting Hamiltonian? Because its ground state is one we can actually prepare: $X|+\rangle = +1\,|+\rangle$ (the $|+\rangle$ superposition state from Part 1 is an eigenstate of $X$ with eigenvalue $+1$), so $-X$ assigns $|+\rangle$ energy $-1$, and since the eigenvalues of $-X$ are $\pm 1$, that is the minimum.

Two API notes: `schedule[t]` returns the effective Hamiltonian at time $t$ (nice for spot-checks), and `len(schedule)` is the number of simulation time steps, $\lfloor T/\mathrm{dt}\rfloor$ (floor division, so $T=10$, $\mathrm{dt}=0.1$ gives 99 steps, not 100). `schedule.draw()` plots the envelopes.

(Filed away for Part 6: this continuous control envelope is the closest thing QiliSDK has to *pulse shaping*. It is Hamiltonian-level control, not waveforms on physical wires, but it is the same engineering instinct.)

In [ ]:
from qilisdk.analog import Schedule

T = 10.0
schedule = Schedule(
    hamiltonians={"driver": -X(0), "problem": Z(0)},
    coefficients={
        "driver":  {(0.0, T): lambda t: 1 - t / T},   # envelope 1 -> 0
        "problem": {(0.0, T): lambda t: t / T},       # envelope 0 -> 1
    },
    dt=0.1,
)
print("total time:", schedule.T, "| time steps:", len(schedule))
print("H(t=0)  :", schedule[0.0])     # - X(0)
print("H(t=5)  :", schedule[5.0])     # -0.5 X(0) + 0.5 Z(0)
print("H(t=10) :", schedule[10.0])    # Z(0)
schedule.draw()

### Evolving a state under the schedule

To actually run the physics we wrap the schedule in the **`AnalogEvolution`** functional (the analog counterpart of Part 2's `DigitalPropagation`) together with an initial state, and execute it on `QiliSim` with the same `Readout` builder as always. `QTensor.uniform(1)` prepares $|+\rangle$, the driver's ground state.

What should come out the other end? The schedule finishes at $H(T) = Z_0$. Careful with a classic trap here: **"ground state" means lowest energy**, and for $+Z$ the eigenvalues are $+1$ for $|0\rangle$ and $-1$ for $|1\rangle$, so the ground state of $+Z_0$ is $|1\rangle$, not $|0\rangle$. If the evolution tracked the ground state the whole way, the final state should be close to $|1\rangle$, i.e. $\langle Z\rangle \approx -1$.

The readout requests **expectation values**: $\langle\psi|\,O\,|\psi\rangle$ for each observable $O$, which (decoded once in Part 1) is just row vector @ matrix @ column vector = one number, the average measured value. With the default `nshots=0` it is computed *exactly* from the state, no sampling noise, so the numbers below are deterministic.

In [ ]:
from qilisdk.functionals import AnalogEvolution
from qilisdk.backends import QiliSim
from qilisdk.readout import Readout
from qilisdk.core import QTensor

result = QiliSim().execute(
    AnalogEvolution(schedule, QTensor.uniform(1)),   # start in |+>, the ground state of -X
    Readout().with_expectation(observables=[Z(0), X(0)]),
)
print("[<Z>, <X>] at t=T:", np.round(result.get_expectation_values(), 4))
# <Z> lands near -1: the state ended close to |1>, the ground state of Z(0)

## 3.2b · Interlude: simulating magnetism

Before we anneal, here is one demonstration of analog evolution used for its original purpose. When Feynman proposed quantum computers in 1982, this was the intended application: simulating quantum matter, because classical simulation runs into the $2^n$ memory wall from Part 1. Simulating quantum magnets is exactly what today's analog quantum simulators (Google, QuEra, Pasqal) are built to do.

Our miniature magnet is a chain of 4 spins with an **XY hopping** interaction between neighbors,

$$H = \sum_{i=0}^{2} \tfrac{1}{2}\left(X_i X_{i+1} + Y_i Y_{i+1}\right).$$

This interaction conserves the number of flipped spins, so a single flip cannot disappear; it can only *move* to a neighbor. We prepare $|1000\rangle$ (leftmost spin flipped, the other three at rest), let the Schrödinger equation run, and track $\langle Z_i\rangle$ on every site: $+1$ means "at rest", $-1$ means "flipped".

Note what we do *not* write: no gates, no algorithm, no update rule. We specify the energy function and an initial state, and the physics (here, `QiliSim` integrating the Schrödinger equation) produces the dynamics. The flip then travels down the chain like a wave: by $t \approx 2.6$ it reaches the far end ($\langle Z_3\rangle \approx -0.85$, bottoming out near $-0.94$), and the run ends with site 3 flipped ($\langle Z_3\rangle \approx -0.88$) while the other sites return most of the way to rest.

In [ ]:
from qilisdk.core import ket

n = 4
hopping = sum(0.5 * (X(i) * X(i + 1) + Y(i) * Y(i + 1)) for i in range(n - 1))

spin_wave = Schedule(hamiltonians={"H": hopping}, total_time=3.0, dt=0.05)
result = QiliSim().execute(
    AnalogEvolution(schedule=spin_wave, initial_state=ket(1, 0, 0, 0),
                    store_intermediate_results=True),
    Readout().with_expectation(observables=[Z(i) for i in range(n)]),
)

history = np.array(result.get_intermediate_expectation_values())   # one row per time step, one column per site
print("time steps:", history.shape[0])                             # 59 (floor division again)
print("final <Z_i> per site:", np.round(history[-1], 4))

times = np.array(spin_wave.tlist)
for site in range(n):
    plt.plot(times, history[:, site], label=rf"$\langle Z_{site}\rangle$")
plt.xlabel("time")
plt.ylabel("expectation value")
plt.title("A spin flip travels down a 4-site chain")
plt.legend()
plt.show()

## 3.3 · Quantum annealing: solve the schedule

Now assemble the pieces. From §3.1b we have a problem Hamiltonian whose ground state is a valid schedule. From §3.2 we know how to evolve a state under a changing Hamiltonian. The missing ingredient is a theorem:

**The adiabatic theorem.** If a system starts in the ground state of its Hamiltonian, and the Hamiltonian changes *slowly enough*, the system stays in the instantaneous ground state the whole way. ("Adiabatic" is physics jargon for "slow enough that the system keeps up.")

That turns ground-state search into a recipe, **quantum annealing**:

1. **Start easy.** Use the driver $H_d = -\sum_i X_i$. Its ground state is every qubit in $|+\rangle$, which is the uniform superposition over all $2^n$ bitstrings (`QTensor.uniform(n)`): every possible schedule at once, and trivial to prepare.
2. **End hard.** The problem Hamiltonian $H_p$ encodes the cost function; its ground state is the answer.
3. **Interpolate slowly.** Run $H(t) = (1 - s)\,H_d + s\,H_p$ with $s = t/T$ going $0 \to 1$. Slowly enough, and the easy ground state is carried continuously into the answer.

`Schedule.linear(H_d, H_p, total_time, dt)` builds exactly this ramp (labeling the two terms `'driver'` and `'problem'`). Anneal too fast and the state is left behind in excited states, i.e. wrong answers with real probability. That failure mode is the subject of Exercise 3.2.

For the readout we request **state tomography**: reconstructing the full final state, which is exact and free on a simulator (on hardware it is expensive), so the probabilities we print are deterministic. In the same run we also request the conflict-pair correlators $\langle Z_i Z_j\rangle$ at every intermediate time step, so we can track how each conflict is resolved during the anneal.

A reality check: at 4 qubits, the brute force you wrote in Exercise 3.1 is instant, and this anneal is overkill. The point is that the *workflow* is identical on D-Wave-class hardware with thousands of qubits, where $2^n$ rules brute force out.

In [ ]:
# The problem Hamiltonian from Exercise 3.1 (rebuilt here so this cell stands alone)
problem = sum(Z(i) * Z(j) for i, j in conflicts)
driver = sum(-X(i) for i in range(len(talks)))

anneal = Schedule.linear(driver, problem, total_time=30, dt=0.1)
result = QiliSim().execute(
    AnalogEvolution(schedule=anneal, initial_state=QTensor.uniform(len(talks)),
                    store_intermediate_results=True),
    Readout().with_state_tomography()
             .with_expectation(observables=[Z(i) * Z(j) for i, j in conflicts]),
)

probs = result.state_tomography.probabilities
top = sorted(probs.items(), key=lambda kv: -kv[1])[:4]
print("most probable final bitstrings:")
for bits, p in top:
    print(f"  {bits}  p = {p:.3f}")

print()
print("decoded schedules (first character = qubit 0 = first talk):")
for bits, p in top:
    if p < 0.01:
        continue
    slots = ", ".join(f"{talk}: {'morning' if b == '0' else 'afternoon'}"
                      for talk, b in zip(talks, bits))
    print(f"  p = {p:.3f} -> {slots}")

The annealer handed back exactly the two schedules your brute force certified in Exercise 3.1, each with probability $0.500$: measure the final state once and you get a valid schedule.

Why *two* answers? Look at the cost function: it only ever compares talks pairwise ($Z_i Z_j$), so nothing in the energy distinguishes morning from afternoon. Swap every talk's slot and every $z_i z_j$ product is unchanged: the problem has a **mirror symmetry**, and the annealer, with no reason to prefer either mirror image, ends in an equal superposition of both. Symmetries in your cost function surface as multiple equally good answers; that is expected, not a bug, and Exercise 3.2 breaks the symmetry deliberately.

The intermediate correlators tell the story of *how* the answer emerged. Each conflict pair starts at $\langle Z_i Z_j\rangle = 0$ (the uniform superposition holds no correlations at all) and is driven to $-1$ (the two talks are in opposite slots with certainty). The four curves move together: the anneal resolves all four conflicts as one continuous process, not one at a time.

In [ ]:
trajectory = np.array(result.get_intermediate_expectation_values())   # one row per time step
times = np.array(anneal.tlist)
for k, (i, j) in enumerate(conflicts):
    plt.plot(times, trajectory[:, k], label=rf"$\langle Z_{i} Z_{j}\rangle$")
plt.xlabel("time")
plt.ylabel("conflict-pair correlator")
plt.title("Annealing: every conflicting pair is driven to opposite slots")
plt.legend()
plt.show()

### 🧩 Exercise 3.2 · The annealer's speed knob

**Setup (given):** suppose the "Async Python" speaker can only make the morning. We pin talk 0 by adding a bias term $-0.8\,Z_0$ to the problem: it lowers the energy when $z_0 = +1$ (bit `0`, morning). This breaks the mirror symmetry, so there is now exactly one best schedule, `0101`.

**Goal:** measure how the probability of finding that schedule depends on annealing time, and find (roughly) the cheapest `total_time` that gets you above 0.9.

**Why it matters:** the adiabatic theorem says "slowly enough" but never says how slow. On production annealers this is a real, billable knob (D-Wave exposes annealing time in microseconds, and slow anneals cost throughput). The practical answer is always empirical: sweep the knob and look, which is exactly what you are about to do.

**Your task:** for each `total_time` in `(0.5, 2, 5, 20)`:

1. build a linear anneal from `driver` to `pinned` with that `total_time`, keeping `dt=0.1`;
2. evolve from the uniform superposition over 4 qubits and read the final state with state tomography (exact and deterministic, so your numbers are stable);
3. print the probability of the winning bitstring `"0101"` (the tomography probabilities are a dict keyed by bitstring, so a plain dictionary lookup does the reading).

Then eyeball the trend: which is roughly the cheapest sweep value that clears 0.9?

In [ ]:
pinned = problem - 0.8 * Z(0)   # given: extra energy reward for talk 0 in the morning

# TODO: for each total_time in (0.5, 2, 5, 20):
#   1. build a linear anneal from `driver` to `pinned` with that total_time (dt=0.1)
#   2. run an AnalogEvolution from QTensor.uniform(len(talks))
#      with a state-tomography readout
#   3. print the probability of "0101" in the final state

## 3.4 · (Optional) Bridging back to circuits: Trotterization

Analog and digital are two views of the same physics, and QiliSDK can translate between them. **Trotterization** is the numerical-methods move you already know from Euler integration: chop a continuous process into many small discrete steps. Here, the continuous evolution $e^{-iHt}$ is approximated by a product of short evolutions, and each short step is then expressed as a handful of gates. The net effect: an analog `Schedule` compiles into a digital `Circuit`.

`TrotterizedSchedule` is a **circuit builder** that does this automatically. (Part 4 introduces the proper term "ansatz" for parameterized circuit builders like this one; we save that word until then.) It is also a preview of Part 6, where we keep lowering circuits toward what hardware actually executes.

> ⚠️ Import-order quirk: import `qilisdk.digital` **before** the trotterization machinery, or you hit a circular-import error. The cell below imports in the safe order.

In [ ]:
from qilisdk.digital import Circuit, TrotterizedSchedule   # import digital first (import-order quirk)

small = Schedule.linear(X(0) + X(1), Z(0) * Z(1), total_time=2.0, dt=0.5)
circuit_builder = TrotterizedSchedule(schedule=small, trotter_steps=1)
print("Trotterized into a circuit on", circuit_builder.nqubits,
      "qubits with", len(circuit_builder.gates), "gates")
circuit_builder.draw()

## Recap & what's next

- A **Hamiltonian** is a cost function over bitstrings: built from Pauli strings, diagonal when it is $Z$-only, and its **ground state is the argmin**. You certified an encoding classically before trusting any quantum machinery, a habit that carries over to every later problem.
- A **`Schedule`** ($H(t) = \sum_k c_k(t)\,H_k$) is time-dependent control, and **`AnalogEvolution`** runs it through the same `Backend.execute(functional, readout)` pattern as every other QiliSDK program.
- **Quantum annealing**: start in the easy ground state of a driver, interpolate slowly to the problem Hamiltonian, and read the answer as a bitstring. It scheduled four conference talks and returned both valid schedules.
- **Annealing speed is a tunable parameter.** The adiabatic theorem's "slowly enough" is an empirical, problem-dependent value, which you measured: anneal too fast and probability is left behind in wrong answers.
- Analog evolution can be **Trotterized** back into a circuit when a gate-based target is required.

**Part 4, Variational Algorithms & Optimization Models:** here we encoded talk scheduling by hand; Part 4's `Model` layer automates the problem-to-Hamiltonian translation for weighted problems with constraints. And instead of evolving in time, we will *train* parameterized circuits: VQE computes a molecule's ground-state energy, the standard first exercise in quantum chemistry, first run on real hardware in 2016.